# Understanding Intraday Bar Timestamps and Close Prices

> This notebook is for educational purposes only. It is not financial advice. The data shown here is synthetic and is used only to demonstrate timestamp conventions in intraday bar data.

## Introduction

Before creating a trading strategy, we need to understand what each timestamp means.

With intraday candle data, a timestamp often does not represent a single instant in isolation.

Instead, it represents a time interval.

For example, a 5-minute candle labelled:

```text
2024-01-01 00:00:00
```

may represent the period:

```text
00:00 to 00:05
```

The open price belongs to the start of the interval.

The close price belongs to the end of the interval.

So if the timestamp is the left bound of the bar, the close price of the candle labelled `00:00` is effectively the price at `00:05`.

This distinction matters when calculating returns, resampling data, and avoiding look-ahead bias.

## Import Libraries

We use:

- `pandas` for time series data and resampling
- `numpy` for synthetic price generation
- `matplotlib` for visualisation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 6)

: 

## Create Synthetic 5-Minute EUR/USD Data

The lecture demonstrates this using Oanda API data.

To keep this notebook self-contained, we will create synthetic 5-minute EUR/USD-style data for one day.

Each row will represent a 5-minute candle.

The timestamp will represent the start of the candle.

In [ ]:
np.random.seed(42)

times = pd.date_range(
    start="2024-01-01 00:00:00",
    end="2024-01-01 23:55:00",
    freq="5min"
)

log_returns = np.random.normal(
    loc=0.0,
    scale=0.0002,
    size=len(times)
)

close_prices = 1.2160 * np.exp(log_returns.cumsum())

data = pd.DataFrame(
    data={"close": close_prices},
    index=times
)

data.index.name = "time"

data.head()

The first timestamp is `00:00`.

Under the left-bound convention, this row represents the 5-minute bar from:

```text
00:00 to 00:05
```

The close price shown on that row is the close at the end of that bar.

So it is effectively the price at `00:05`.

## Add Explicit Bar Start and Bar End Columns

To make the convention clear, add two helper columns:

- `bar_start`
- `bar_end`

In [ ]:
bar_example = data.head(6).copy()

bar_example["bar_start"] = bar_example.index
bar_example["bar_end"] = bar_example.index + pd.Timedelta(minutes=5)

bar_example[["bar_start", "bar_end", "close"]]

This table shows that the timestamp is the start of the bar.

The close price belongs to the end of the interval.

For example, the row labelled `00:00` covers:

```text
00:00 to 00:05
```

So its close price is the price at the end of the candle, not the price at the very beginning.

## Explain the 6-Hour EUR/USD Example

The same idea applies to the 6-hour EUR/USD data from the previous notebook.

If a 6-hour bar is labelled:

```text
2018-01-02 04:00:00
```

and the timestamp is the left bound, then the bar covers:

```text
04:00 to 10:00
```

The close price on that row is therefore the close at the end of the 6-hour candle.

Effectively, it is the price at `10:00`, even though the row is labelled `04:00`.

This is why we should think of candle timestamps as interval labels, not always as exact single moments.

## Why Returns Belong to Periods, Not Isolated Timestamps

Returns are also period-based.

A return cannot be calculated from one single price alone.

A return compares two prices:

```text
previous close to current close
```

For example, the return on the candle labelled `00:05` compares:

```text
close of 00:00 bar
close of 00:05 bar
```

That means the first row has no return because there is no previous close.

In [ ]:
data["returns"] = np.log(data["close"] / data["close"].shift(1))

data.head()

The first return is missing because no previous close exists.

Starting from the second row, each return measures the change from the previous candle close to the current candle close.

## Visualise the Close Price Series

In [ ]:
data["close"].plot()

plt.title("Synthetic 5-Minute EUR/USD Close Prices")
plt.xlabel("Time")
plt.ylabel("Close Price")
plt.grid(True)
plt.show()

The line connects the close prices of each 5-minute candle.

Even though the index labels are the bar start times, the close values represent the end of each bar.

## Resample 5-Minute Data to 20-Minute Bars

Pandas also follows a similar interval idea when resampling.

Suppose we want to convert 5-minute close prices into 20-minute close prices.

We can use:

```python
data["close"].resample("20min").last()
```

This takes the last available close price inside each 20-minute interval.

In [ ]:
close_20min = data["close"].resample("20min").last()

close_20min.head()

The first 20-minute bar labelled `00:00` covers:

```text
00:00 to 00:20
```

The value is the last 5-minute close inside that interval.

That is effectively the close at the end of the 20-minute bar.

## Show Where the First 20-Minute Close Comes From

Let’s inspect the original 5-minute rows that belong to the first 20-minute interval.

In [ ]:
data.loc["2024-01-01 00:00:00":"2024-01-01 00:15:00", ["close"]]

In [ ]:
close_20min.head(1)

The first 20-minute interval labelled `00:00` contains the 5-minute bars labelled:

```text
00:00
00:05
00:10
00:15
```

The resampled 20-minute close is the last close inside that interval.

This corresponds to the end of the 20-minute period.

## Demonstrate pandas `label` Behaviour

By default, many intraday resampling operations in pandas label intervals using the left edge.

That means the interval from:

```text
00:00 to 00:20
```

is labelled as:

```text
00:00
```

But we can also label the result using the right edge of each interval.

In [ ]:
close_20min_left_label = data["close"].resample("20min", label="left").last()
close_20min_right_label = data["close"].resample("20min", label="right").last()

comparison = pd.DataFrame({
    "left_label": close_20min_left_label.head(5),
    "right_label": close_20min_right_label.head(5)
})

comparison

The values can represent the same interval closes, but the labels shift depending on whether we label the interval by its left edge or right edge.

Left label:

```text
00:00 label = interval 00:00 to 00:20
```

Right label:

```text
00:20 label = interval 00:00 to 00:20
```

This is why understanding timestamp conventions is important.

## Resample to 6-Hour Bars

Now apply the same idea to 6-hour bars, which matches the previous notebook.

In [ ]:
close_6h_left = data["close"].resample("6h", label="left").last()
close_6h_right = data["close"].resample("6h", label="right").last()

six_hour_comparison = pd.DataFrame({
    "close_label_left": close_6h_left,
    "close_label_right": close_6h_right
})

six_hour_comparison.head()

A 6-hour bar labelled `00:00` with `label="left"` covers:

```text
00:00 to 06:00
```

The close value represents the end of that interval.

With `label="right"`, the same interval would be labelled:

```text
06:00
```

The underlying close value can be the same, but the timestamp label is different.

## Practical Implication for the Contrarian Strategy

This matters for strategy testing.

If a 6-hour bar is labelled `04:00` and the close price represents the close of the interval from `04:00` to `10:00`, then we cannot use that close price to make a trade at `04:00`.

We only know the close price after the bar has completed.

So the earliest we can act on that information is the next bar.

That is why strategy returns should usually use shifted positions:

```python
strategy_returns = position.shift(1) * returns
```

This ensures that the strategy uses information from completed candles only.

## Visual Timeline

```text
5-minute bars with left-bound labels:

Label       Bar interval       Close known at
00:00      00:00 - 00:05       00:05
00:05      00:05 - 00:10       00:10
00:10      00:10 - 00:15       00:15
00:15      00:15 - 00:20       00:20

20-minute resampled bar:

Label       Bar interval       Close known at
00:00      00:00 - 00:20       00:20
```

The label is not always the time when the close was known.

The close is only known at the end of the bar.

## Key Takeaways

In this notebook, we clarified how intraday bar timestamps work.

Key points:

- Intraday candle timestamps often label time intervals.
- With left-bound labelling, the timestamp marks the start of the bar.
- The close price belongs to the end of the bar.
- A 5-minute bar labelled `00:00` covers `00:00` to `00:05`.
- A 6-hour bar labelled `04:00` covers `04:00` to `10:00`.
- Returns are period-based, not single-timestamp values.
- The first return is missing because there is no previous close.
- Pandas `resample()` can label intervals using the left or right edge.
- Strategy signals should be shifted before calculating returns to avoid using information before it is known.
- The next notebook will use these prepared returns to create a simple contrarian strategy.